In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AzoReduction(MorphingOperator):
    def __init__(self):
        super(AzoReduction, self).__init__()
        self._name = "Azo Reduction (Phase I - Safe Fragmentation)"
        self._matches = []
        self.AZO_PATTERN = Chem.MolFromSmarts("[C,c]-[N;!R]=[N;!R]-[C,c]")

    def setOriginal(self, mol):
        super(AzoReduction, self).setOriginal(mol)
        self._matches = []

        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        matches = rdkit_mol.GetSubstructMatches(self.AZO_PATTERN)
        for match in matches:
            self._matches.append((match[0], match[1], match[2], match[3]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        edit_mol = Chem.RWMol(rdkit_mol)
        c1_idx, n1_idx, n2_idx, c2_idx = random.choice(self._matches)

        try:
            edit_mol.RemoveBond(n1_idx, n2_idx)

            for idx in [n1_idx, n2_idx]:
                atom = edit_mol.GetAtomWithIdx(idx)
                atom.SetFormalCharge(0)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                for prop in list(atom.GetPropNames()):
                    atom.ClearProp(prop)

            new_mol = edit_mol.GetMol()
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            fragments = rdmolops.GetMolFrags(new_mol, asMols=True)
            if not fragments:
                return MolpherMol(other=rdkit_mol)

            chosen_frag = random.choice(fragments)
            
            Chem.AssignStereochemistry(chosen_frag, cleanIt=True, force=True)
            return MolpherMol(other=chosen_frag)
                
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

azo_op = AzoReduction()

print("\n=== RUNNING AZO REDUCTION TEST (FRAGMENTATION) ===")
azo_molecule = "C1=CC=C(C=C1)/N=N/C2=CC=CC=C2" # Αζωβενζόλιο
mol_azo = MolpherMol(azo_molecule)
azo_op.setOriginal(mol_azo)
prod_azo = azo_op.morph()
print(f"Αζωβενζόλιο\n  SRC: {mol_azo.getSMILES()}\n  TRG: {prod_azo.getSMILES() if prod_azo else 'Failed'}")
print("==================================================")


=== RUNNING AZO REDUCTION TEST (FRAGMENTATION) ===
Αζωβενζόλιο
  SRC: C1=CC=C(N=NC2=CC=CC=C2)C=C1
  TRG: NC1=CC=CC=C1
